# Offline Automated Code Review Board Demo

In [ ]:
from pathlib import Path
import asyncio
from core.state import ChangeState
from agents.diff_agent import DiffAgent
from agents.style_agent import StyleAgent
from agents.security_agent import SecurityAgent
from agents.test_coverage_agent import TestCoverageAgent
from agents.reviewer_manager import ReviewerManager

diff = Path('../data/sample_diff.patch').read_text()
state = ChangeState(diff_text=diff)


In [ ]:
async def demo():
    await DiffAgent().execute(state)
    # Genuine fan-out: specialists run concurrently.
    await asyncio.gather(StyleAgent().execute(state), SecurityAgent().execute(state), TestCoverageAgent().execute(state))
    return await ReviewerManager().execute(state)

report = await demo()
print('Tools:', state.tools_invoked)
for agent, findings in state.findings_by_agent.items():
    print(agent, [f.title for f in findings])
print('Recommendation:', report.recommendation)
print('Trace:', state.trace)
